## 1. Installation des dépendances

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'requests', 'beautifulsoup4', '-q'])

0

## 2. Imports

In [2]:
import json
import csv
import time
import os

import requests
from bs4 import BeautifulSoup

## 3. Configuration

In [3]:
BASE_URL = 'https://www.entreparticuliers.com'

TRANSACTION_URLS = {
    'vente':    BASE_URL + '/annonces-immobilieres/ventes/',
    'location': BASE_URL + '/annonces-immobilieres/location/',
}

HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
        'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36'
    ),
    'Accept-Language': 'fr-FR,fr;q=0.9',
}

MAX_PAGES  = 20    
DELAY      = 1.5  
OUTPUT_DIR = '../data'

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 4. Fonctions de scraping

### 4.1 Requête HTTP

In [4]:
def fetch_page(url: str):

    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        response.raise_for_status()
        return response.text
    except requests.RequestException as e:
        print(f'  [ERREUR] {url} → {e}')
        return None

html_test = fetch_page(TRANSACTION_URLS['vente'] + '?page=1')

### 4.2 Extraction du JSON embarqué

In [5]:
def extract_annonces_from_html(html: str):

    soup = BeautifulSoup(html, 'html.parser')

    for script in soup.find_all('script'):
        if not script.string:
            continue
        try:
            data = json.loads(script.string)
        except (json.JSONDecodeError, ValueError):
            continue

        for key, value in data.items():
            if not isinstance(value, dict) or 'b' not in value:
                continue
            block = value['b']
            if (
                isinstance(block, dict)
                and block.get('@type') == 'hydra:Collection'
                and 'annonces' in block.get('@id', '')
                and 'hydra:member' in block
            ):
                return block['hydra:member']

    return []

raws_test = extract_annonces_from_html(html_test)

### 4.3 Normalisation d'une annonce

In [6]:
def parse_annonce(raw: dict, type_transaction: str):

    try:
        prix    = raw.get('prix')
        surface = raw.get('surface')
        if prix is None or surface is None:
            return None

        commune   = raw.get('commune', {})
        bien_type = raw.get('bienType', {})
        rubrique  = raw.get('rubrique', {})

        return {
            'type_transaction': rubrique.get('slug', type_transaction),
            'type_bien':        bien_type.get('label', ''),
            'prix':             prix,
            'surface':          surface,
            'nb_pieces':        raw.get('piecesnb', None),
            'ville':            commune.get('label', ''),
            'code_postal':      commune.get('codePostal', ''),
            'description':      '',  # non disponible sans rendu JS
            'source':           'entreparticuliers',
        }
    except Exception:
        return None

exemple = parse_annonce(raws_test[0], 'vente') if raws_test else None

## 5. Lancement du scraping

In [8]:
all_annonces = []

for transaction, base_url in TRANSACTION_URLS.items():
    print(f"\n{'='*50}")
    print(f"Type : {transaction.upper()} — {MAX_PAGES} pages max")
    print(f"{'='*50}")

    for page in range(1, MAX_PAGES + 1):
        url = f'{base_url}?page={page}'
        print(f'  Page {page}/{MAX_PAGES}...', end=' ')

        html = fetch_page(url)
        if not html:
            print('ERREUR — arrêt')
            break

        raws   = extract_annonces_from_html(html)
        if not raws:
            print('aucune annonce — fin de pagination')
            break

        parsed = [parse_annonce(r, transaction) for r in raws]
        valid  = [a for a in parsed if a is not None]
        all_annonces.extend(valid)
        print(f'{len(valid)} annonces  (cumul : {len(all_annonces)})')

        time.sleep(DELAY)

print(f"\n✅ Scraping terminé — {len(all_annonces)} annonces collectées")


Type : VENTE — 20 pages max
  Page 1/20... 12 annonces  (cumul : 12)
  Page 2/20... 12 annonces  (cumul : 24)
  Page 3/20... 12 annonces  (cumul : 36)
  Page 4/20... 12 annonces  (cumul : 48)
  Page 5/20... 12 annonces  (cumul : 60)
  Page 6/20... 12 annonces  (cumul : 72)
  Page 7/20... 12 annonces  (cumul : 84)
  Page 8/20... 12 annonces  (cumul : 96)
  Page 9/20... 12 annonces  (cumul : 108)
  Page 10/20... 12 annonces  (cumul : 120)
  Page 11/20... 12 annonces  (cumul : 132)
  Page 12/20... 12 annonces  (cumul : 144)
  Page 13/20... 12 annonces  (cumul : 156)
  Page 14/20... 12 annonces  (cumul : 168)
  Page 15/20... 12 annonces  (cumul : 180)
  Page 16/20... 12 annonces  (cumul : 192)
  Page 17/20... 12 annonces  (cumul : 204)
  Page 18/20... 12 annonces  (cumul : 216)
  Page 19/20... 12 annonces  (cumul : 228)
  Page 20/20... 12 annonces  (cumul : 240)

Type : LOCATION — 20 pages max
  Page 1/20... 12 annonces  (cumul : 252)
  Page 2/20... 12 annonces  (cumul : 264)
  Page 3/20.

## 6. Sauvegarde des données

Export vers **JSON** (format brut complet) et **CSV** (format tabulaire pour Spark).

In [9]:
# --- JSON ---
json_path = os.path.join(OUTPUT_DIR, 'entreparticuliers_raw.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(all_annonces, f, ensure_ascii=False, indent=2)
print(f'JSON sauvegardé → {json_path}')

# --- CSV ---
csv_path = os.path.join(OUTPUT_DIR, 'entreparticuliers_raw.csv')
if all_annonces:
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=all_annonces[0].keys())
        writer.writeheader()
        writer.writerows(all_annonces)
    print(f'CSV sauvegardé  → {csv_path}')

print(f'\nTotal : {len(all_annonces)} annonces exportées')

JSON sauvegardé → ../data/entreparticuliers_raw.json
CSV sauvegardé  → ../data/entreparticuliers_raw.csv

Total : 480 annonces exportées
